# Notebook 3 — the submission run

Everything notebook 2 left open, at the level a DMKD referee will ask for.

| added here | why the referee asks |
|---|---|
| 9 datasets across 5 domains | five was thin for an empirical claim |
| full size, no simplex subsampling | notebook 2 capped `congress-bills` at 3k of 85k simplices |
| a **learned** predictor, strict leave-one-graph-out | a perfect oracle is not actionable; the whole claim now rests on learnability |
| feature ablation: degrees/cores, + higher-order, + covered-neighbour | shows *which* signal carries the new primitive |
| 50 permutation seeds, CIs, permutation $p$-value | the 100% structural share rested on one seed |
| Monte-Carlo validation of the analytic probabilities | the speedups are computed, not simulated — prove they are right |
| cap-sensitivity and ridge-$\lambda$ checks | rules out artefacts of our own choices |

Runtime is the cost of honesty: expect roughly 20–60 minutes on a standard Colab CPU runtime,
dominated by `congress-bills`. Run it once and keep the CSVs.

**Reminder on what is being separated.** `loc_x` is the gain from localization alone
($\hat w \equiv 0$: the local normalizer $D_x$ instead of the global $\sqrt{2m}$ rule). Every
predictor is then scored *on top of* that, as `..._over_loc`. Only the second column is about
prediction, and only it belongs in the paper's headline claim.

## 0. Library

In [ ]:
"""Paper-grade machinery for prediction-augmented counting of cross-hyperedge triangles.

Everything that is evaluated many times (many predictors, many permutation seeds) is
vectorised over precomputed triangle triples, so the expensive structural pass runs once.
"""
import numpy as np


def _bits_to_idx(x, nbytes):
    """Indices of the set bits of a Python int, fast."""
    if x == 0:
        return np.empty(0, dtype=np.int32)
    b = np.frombuffer(x.to_bytes(nbytes, 'little'), dtype=np.uint8)
    return np.flatnonzero(np.unpackbits(b, bitorder='little')).astype(np.int32)


def _popcount(x):
    try:
        return x.bit_count()
    except AttributeError:
        return bin(x).count('1')


class CrossHG:
    """Projected graph of a hypergraph + exact cross-triangle structure.

    Cross triangle: {u,v,w} pairwise adjacent in the projection, with no single
    hyperedge containing all three.  Canonical discovery follows the Fichtenberger-Peng
    path used by PredCount: order the three vertices by (degree, id) as x < y < z,
    base edge {x,y}, pivot x, closing vertex z.
    """

    def __init__(self, hyperedges, max_size=25, max_triples=4_000_000, seed=0, verbose=True):
        hs = [tuple(sorted(set(h))) for h in hyperedges]
        hs = [h for h in hs if 2 <= len(h) <= max_size]
        hs = list(dict.fromkeys(hs))
        verts = sorted({v for h in hs for v in h})
        idx = {v: i for i, v in enumerate(verts)}
        self.n = n = len(verts)
        self.nbytes = (n + 7) // 8
        self.hs = [tuple(idx[v] for v in h) for h in hs]
        self.hbits = [sum(1 << a for a in h) for h in self.hs]
        self.hsize = np.array([len(h) for h in self.hs], dtype=np.int32)

        memb = [[] for _ in range(n)]
        adj = [0] * n
        for hid, h in enumerate(self.hs):
            bits = self.hbits[hid]
            for a in h:
                memb[a].append(hid)
                adj[a] |= bits & ~(1 << a)
        self.memb = [set(m) for m in memb]
        self.adj = adj

        # ---- edges, CSR adjacency, edge ids
        nbr = [_bits_to_idx(adj[u], self.nbytes) for u in range(n)]
        self.deg = np.array([len(x) for x in nbr], dtype=np.int64)
        self.indptr = np.zeros(n + 1, dtype=np.int64)
        np.cumsum(self.deg, out=self.indptr[1:])
        self.indices = np.concatenate(nbr) if n else np.empty(0, np.int32)

        eu, ev = [], []
        for u in range(n):
            w = nbr[u][nbr[u] > u]
            eu.append(np.full(len(w), u, dtype=np.int32))
            ev.append(w)
        self.eu = np.concatenate(eu) if n else np.empty(0, np.int32)
        self.ev = np.concatenate(ev) if n else np.empty(0, np.int32)
        self.m = len(self.eu)
        self.eid = {}
        for i in range(self.m):
            self.eid[(int(self.eu[i]), int(self.ev[i]))] = i
        # edge id for every CSR slot
        self.slot_eid = np.empty(len(self.indices), dtype=np.int64)
        for u in range(n):
            a, b = self.indptr[u], self.indptr[u + 1]
            for k in range(a, b):
                v = int(self.indices[k])
                self.slot_eid[k] = self.eid[(u, v)] if u < v else self.eid[(v, u)]

        self._core_numbers()
        self.rank = np.empty(n, dtype=np.int64)
        order = np.lexsort((np.arange(n), self.deg))
        self.rank[order] = np.arange(n)

        self._cross_and_triples(max_triples, seed, verbose)
        self._features()

    # ------------------------------------------------------------------ structure
    def _core_numbers(self):
        n = self.n
        deg = self.deg.copy()
        core = np.zeros(n, dtype=np.int64)
        removed = np.zeros(n, dtype=bool)
        maxd = int(deg.max()) if n else 0
        buckets = [set() for _ in range(maxd + 1)]
        for v in range(n):
            buckets[deg[v]].add(v)
        k, i = 0, 0
        for _ in range(n):
            while i <= maxd and not buckets[i]:
                i += 1
            if i > maxd:
                break
            v = buckets[i].pop()
            k = max(k, i)
            core[v] = k
            removed[v] = True
            for j in range(self.indptr[v], self.indptr[v + 1]):
                w = int(self.indices[j])
                if removed[w]:
                    continue
                d = int(deg[w])
                buckets[d].discard(w)
                deg[w] = d - 1
                buckets[d - 1].add(w)
                if d - 1 < i:
                    i = d - 1
        self.core = core
        self.kappa = int(k)

    def _cross_and_triples(self, max_triples, seed, verbose):
        """Per-edge cross-triangle counts (exact) and the triple arrays (possibly sampled)."""
        nb = self.nbytes
        t = np.zeros(self.m, dtype=np.int64)
        covn = np.zeros(self.m, dtype=np.int64)     # covered common neighbours
        nhy = np.zeros(self.m, dtype=np.int64)      # hyperedges containing the edge
        hsum = np.zeros(self.m, dtype=np.int64)     # sum over those of (|h|-2)
        hmax = np.zeros(self.m, dtype=np.int64)
        covbits = [0] * self.m
        for i in range(self.m):
            u, v = int(self.eu[i]), int(self.ev[i])
            common = self.adj[u] & self.adj[v]
            S = self.memb[u] & self.memb[v]
            cov = 0
            for h in S:
                cov |= self.hbits[h]
            covbits[i] = cov
            nhy[i] = len(S)
            if S:
                sz = self.hsize[list(S)]
                hsum[i] = int((sz - 2).sum())
                hmax[i] = int(sz.max())
                covn[i] = _popcount(cov & common)
            t[i] = _popcount(common & ~cov)
        self.t = t
        self.covn, self.nhy, self.hsum, self.hmax = covn, nhy, hsum, hmax
        self.n_cross_total = int(t.sum()) // 3

        # triples: each cross triangle is generated once, from its canonical base edge
        target = self.n_cross_total if self.n_cross_total else 1
        p = min(1.0, max_triples / target)
        rng = np.random.default_rng(seed)
        keep = np.ones(self.m, dtype=bool) if p >= 1.0 else (rng.random(self.m) < p)
        self.triple_scale = 1.0 / p
        self.triple_fraction = p

        base, pivot, xz = [], [], []
        rank = self.rank
        for i in np.flatnonzero(keep):
            i = int(i)
            if t[i] == 0:
                continue
            u, v = int(self.eu[i]), int(self.ev[i])
            cross = (self.adj[u] & self.adj[v]) & ~covbits[i]
            z = _bits_to_idx(cross, nb)
            hi = max(rank[u], rank[v])
            z = z[rank[z] > hi]
            if not len(z):
                continue
            x, y = (u, v) if rank[u] < rank[v] else (v, u)
            base.append(np.full(len(z), i, dtype=np.int32))
            pivot.append(np.full(len(z), x, dtype=np.int32))
            xz.append(np.array([self.eid[(min(x, int(c)), max(x, int(c)))] for c in z],
                               dtype=np.int32))
        self.base = np.concatenate(base) if base else np.empty(0, np.int32)
        self.pivot = np.concatenate(pivot) if pivot else np.empty(0, np.int32)
        self.xz = np.concatenate(xz) if xz else np.empty(0, np.int32)
        if verbose:
            print(f'    n={self.n} m={self.m} kappa={self.kappa} '
                  f'#cross={self.n_cross_total} triples kept={len(self.base)} '
                  f'({100*self.triple_fraction:.1f}% of edges)')

    def _features(self):
        lg = np.log1p
        du, dv = self.deg[self.eu], self.deg[self.ev]
        cu, cv = self.core[self.eu], self.core[self.ev]
        ku = np.array([len(self.memb[u]) for u in self.eu], dtype=np.int64)
        kv = np.array([len(self.memb[v]) for v in self.ev], dtype=np.int64)
        A = np.column_stack([                       # Array-paper block: degrees + cores
            lg(np.minimum(du, dv)), lg(np.maximum(du, dv)),
            lg(du) + lg(dv), np.abs(lg(du) - lg(dv)),
            lg(np.minimum(cu, cv)), lg(np.maximum(cu, cv)), lg(cu) + lg(cv),
        ])
        B = np.column_stack([                       # cheap higher-order block
            lg(np.minimum(ku, kv)), lg(np.maximum(ku, kv)),
            lg(self.nhy), lg(self.hsum), lg(self.hmax),
        ])
        Cc = np.column_stack([lg(self.covn)])       # covered-neighbour block
        self.F = {'A': A, 'AB': np.hstack([A, B]), 'ABC': np.hstack([A, B, Cc])}
        self.y = np.log1p(self.t.astype(float))

    # ------------------------------------------------------------- success probability
    def _D(self, wp):
        """D_x = sum over neighbours c of (w({x,c})+1), as an array over vertices."""
        seg = wp[self.slot_eid]
        out = np.add.reduceat(seg, self.indptr[:-1])
        return out

    def p_succ(self, w, first_edge_only=False):
        wp = np.asarray(w, dtype=float) + 1.0
        W = wp.sum()
        D = self._D(wp)
        if first_edge_only:
            inner = 1.0 / self.deg[self.pivot]
        else:
            inner = wp[self.xz] / D[self.pivot]
        return float((wp[self.base] * inner).sum()) / W * self.triple_scale

    def p_base(self):
        return self.n_cross_total / (2 * self.m) ** 1.5

    # -------------------------------------------------------------------- predictors
    def w_perfect(self):
        return self.t.astype(float)

    def w_uniform(self):
        return np.zeros(self.m)

    def w_mindeg(self):
        return np.minimum(self.deg[self.eu], self.deg[self.ev]).astype(float)

    def w_permuted(self, w, seed):
        rng = np.random.default_rng(seed)
        return rng.permutation(np.asarray(w, dtype=float))


# ------------------------------------------------------------------------ ridge model

def ridge_fit(X, y, lam=1.0):
    mx, sx = X.mean(0), X.std(0) + 1e-12
    Z = (X - mx) / sx
    my, sy = y.mean(), y.std() + 1e-12
    yz = (y - my) / sy
    Z1 = np.hstack([Z, np.ones((len(Z), 1))])
    A = Z1.T @ Z1 + lam * np.eye(Z1.shape[1])
    A[-1, -1] -= lam                       # do not penalise the intercept
    beta = np.linalg.solve(A, Z1.T @ yz)
    return dict(beta=beta, mx=mx, sx=sx, my=my, sy=sy)


def ridge_predict_weights(model, X):
    Z = (X - model['mx']) / model['sx']
    Z1 = np.hstack([Z, np.ones((len(Z), 1))])
    yz = Z1 @ model['beta']
    y = yz * model['sy'] + model['my']
    return np.clip(np.expm1(np.clip(y, 0, 30)), 0, None)


# ------------------------------------------------------------------------- reporting

def evaluate(hg, n_perm=50, seed=0):
    """Localization, perfect predictor, permutation null, cheap heuristic, first-edge."""
    pb = hg.p_base()
    p_loc = hg.p_succ(hg.w_uniform())
    p_perf = hg.p_succ(hg.w_perfect())
    p_mind = hg.p_succ(hg.w_mindeg())
    p_first = hg.p_succ(hg.w_perfect(), first_edge_only=True)
    perms = np.array([hg.p_succ(hg.w_permuted(hg.w_perfect(), seed + b))
                      for b in range(n_perm)])
    s_perf, s_perm = p_perf / p_loc, perms / p_loc
    gain = s_perf - 1.0
    shares = (s_perf - s_perm) / gain if abs(gain) > 1e-12 else np.full(n_perm, np.nan)
    pval = (1 + (s_perm >= s_perf).sum()) / (1 + n_perm)
    return dict(
        n=hg.n, m=hg.m, kappa=hg.kappa, n_cross=hg.n_cross_total,
        copy_density=hg.n_cross_total / max(hg.m, 1),
        triple_fraction=hg.triple_fraction,
        loc_x=p_loc / pb, perfect_x=p_perf / pb, firstedge_x=p_first / pb,
        perfect_over_loc=s_perf,
        permuted_over_loc_mean=float(s_perm.mean()),
        permuted_over_loc_lo=float(np.percentile(s_perm, 2.5)),
        permuted_over_loc_hi=float(np.percentile(s_perm, 97.5)),
        mindeg_over_loc=p_mind / p_loc,
        mindeg_capture=(p_mind / p_loc - 1) / gain if abs(gain) > 1e-12 else np.nan,
        structural_share=float(np.mean(shares)),
        structural_share_lo=float(np.percentile(shares, 2.5)),
        structural_share_hi=float(np.percentile(shares, 97.5)),
        perm_pvalue=float(pval), n_perm=n_perm,
    )


def monte_carlo_check(hg, w, n_samples=200_000, seed=0):
    """Sample real paths from the weighted sampler; compare hit rate to the analytic p."""
    rng = np.random.default_rng(seed)
    wp = np.asarray(w, dtype=float) + 1.0
    pe = wp / wp.sum()
    D = hg._D(wp)
    picks = rng.choice(hg.m, size=n_samples, p=pe)
    hits = 0
    tset = {}
    for i in picks:
        u, v = int(hg.eu[i]), int(hg.ev[i])
        x, y = (u, v) if hg.rank[u] < hg.rank[v] else (v, u)
        a, b = hg.indptr[x], hg.indptr[x + 1]
        slots = hg.slot_eid[a:b]
        pr = wp[slots] / D[x]
        c = int(hg.indices[a + rng.choice(len(slots), p=pr / pr.sum())])
        if c == u or c == v or hg.rank[c] <= hg.rank[y]:
            continue
        key = (min(x, c), max(x, c))
        if key not in hg.eid or (min(y, c), max(y, c)) not in hg.eid:
            continue
        S = hg.memb[x] & hg.memb[y] & hg.memb[c]
        if not S:
            hits += 1
    return hits / n_samples


## 1. Data — nine hypergraphs, five domains

All from Benson's temporal higher-order collection (`cs.cornell.edu/~arb/data`), hosted on Google
Drive, hence `gdown`. Each archive holds `<name>-nverts.txt` and `<name>-simplices.txt`.

In [ ]:
!pip -q install gdown

import gdown, tarfile, glob, os, time, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 240)

ARB_IDS = {
    'contact-primary-school': '1sBHSEIyvVKavAho524Ro4cKL66W6rn-t',   # contact
    'contact-high-school':    '1VA2P62awVYgluOIh1W4NZQQgkQCBk-Eu',   # contact
    'email-Enron':            '1tTVZkdpgRW47WWmsrdUCukHz0x2M6N77',   # email
    'email-Eu':               '1amLeVudLBDRglCXKlieg6HHE-vu81EVF',   # email
    'NDC-classes':            '1tpDiP1c73O18gCYEx4OI7kx8V_IdYxLt',   # drugs
    'NDC-substances':         '1mGOg0DMh46J2zQdimSXMde1pKNtfAdh8',   # drugs
    'DAWN':                   '1wGwoG7oBWnNN7J9TEpjqNpODbsYfMxp4',   # drugs
    'congress-bills':         '1gH1uJMZpn_SCJSRbORPH4JRQeevLwTyO',   # legislation
    'tags-math-sx':           '1eDevpF6EZs19rLouNpiKGLIlFOLUfKKG',   # tags
    'tags-ask-ubuntu':        '1tb1ZJlXEJnlRkXpTuBZlOqqsFknWCkUV',   # tags  (backup for tags-math-sx)
    # optional, n = 125k vertices: the bitset engine is O(n/64) per operation, so this
    # one is slow.  Add it to WANT only if the run above finished comfortably.
    'threads-ask-ubuntu':     '1ppdJ7CvF_aJ9GLJmVWgKpNL55YVH8MYh',   # threads
}
DOMAIN = {'contact-primary-school':'contact', 'contact-high-school':'contact',
          'email-Enron':'email', 'email-Eu':'email',
          'NDC-classes':'drugs', 'NDC-substances':'drugs', 'DAWN':'drugs',
          'congress-bills':'legislation',
          'tags-math-sx':'tags', 'tags-ask-ubuntu':'tags',
          'threads-ask-ubuntu':'threads'}

WANT = ['contact-primary-school', 'contact-high-school', 'email-Enron', 'email-Eu',
        'NDC-classes', 'NDC-substances', 'DAWN', 'congress-bills',
        'tags-math-sx', 'tags-ask-ubuntu']

def load_simplices(nverts_path, simplices_path):
    sizes = [int(x) for x in open(nverts_path).read().split()]
    flat = [int(x) for x in open(simplices_path).read().split()]
    out, i = [], 0
    for s in sizes:
        out.append(tuple(flat[i:i + s])); i += s
    return out

def _extract(tgz):
    with tarfile.open(tgz) as t:
        try:
            t.extractall('.', filter='data')      # silences the 3.14 deprecation warning
        except TypeError:
            t.extractall('.')

# Google Drive throttles; retry, and fall back to the fuzzy URL form.
def fetch_arb(name, tries=3):
    tgz = f'{name}.tar.gz'
    for k in range(tries):
        try:
            if not os.path.exists(tgz) or os.path.getsize(tgz) < 1000:
                ok = gdown.download(id=ARB_IDS[name], output=tgz, quiet=True)
                if not ok:
                    gdown.download(url=f'https://drive.google.com/uc?id={ARB_IDS[name]}',
                                   output=tgz, quiet=True, fuzzy=True)
            _extract(tgz)
            nv = glob.glob(f'**/{name}-nverts.txt', recursive=True)
            sp = glob.glob(f'**/{name}-simplices.txt', recursive=True)
            return load_simplices(nv[0], sp[0])
        except Exception as e:
            print(f'  {name}: attempt {k+1}/{tries} failed ({type(e).__name__})')
            if os.path.exists(tgz) and os.path.getsize(tgz) < 1000:
                os.remove(tgz)
            time.sleep(5)
    print(f'  {name}: giving up. Download it by hand from '
          f'https://drive.google.com/uc?id={ARB_IDS[name]} , upload the .tar.gz here, '
          f'and re-run this cell.')
    return None

HYPER = {}
for name in WANT:
    h = fetch_arb(name)
    if h is None:
        continue
    uniq = list(dict.fromkeys(tuple(sorted(set(x))) for x in h))
    uniq = [x for x in uniq if 2 <= len(x) <= 25]
    HYPER[name] = uniq
    sz = np.array([len(x) for x in uniq])
    print(f'{name:24s} {len(uniq):>7d} unique simplices  size mean {sz.mean():.1f} max {sz.max()}')

print(f'\nloaded {len(HYPER)} datasets across '
      f'{len({DOMAIN[k] for k in HYPER})} domains')
if 'tags-math-sx' not in HYPER and 'tags-ask-ubuntu' in HYPER:
    print('tags-math-sx unavailable; tags-ask-ubuntu covers the tags domain instead.')

## 2. Build the cross-triangle structure

One structural pass per dataset. `#cross` is exact for every dataset — it comes from a bitset
identity (common neighbours minus the hyperedge-covered part), not from enumeration. Only the
*triple list* used to sum per-copy probabilities is subsampled when a graph has more than
`MAX_TRIPLES` cross triangles; `triple_fraction` records it. Because every predictor is scored on
the *same* triple sample, the reported ratios are unaffected by that subsampling — only the absolute
probabilities would be, and those never appear in a claim.

In [ ]:
MAX_TRIPLES = 4_000_000

HG, build_log = {}, []
for name, h in HYPER.items():
    t0 = time.time()
    print(f'{name} ...')
    HG[name] = CrossHG(h, max_size=25, max_triples=MAX_TRIPLES, seed=0)
    build_log.append(dict(label=name, domain=DOMAIN[name], secs=round(time.time() - t0, 1)))
pd.DataFrame(build_log)

## 3. Table 1 — the main measurement

`perfect_over_loc` is what a perfect predictor adds on top of localization.
`permuted_over_loc` is the same weight multiset shuffled across edges, over 50 seeds, with a 95%
interval. `structural_share` is the fraction of the predictor's gain attributable to knowing *which*
edges are heavy rather than to the shape of the weight distribution — the quantity that came out at
0.093 for ordinary triangles in the Array paper.

In [ ]:
N_PERM = 50
rows = []
for name, hg in HG.items():
    t0 = time.time()
    r = evaluate(hg, n_perm=N_PERM, seed=0)
    r['label'], r['domain'], r['secs'] = name, DOMAIN[name], round(time.time() - t0, 1)
    rows.append(r)
    print(f"{name:24s} loc={r['loc_x']:5.2f}x  pred={r['perfect_over_loc']:5.2f}x  "
          f"perm={r['permuted_over_loc_mean']:.3f}x  share={r['structural_share']:.3f}  "
          f"p={r['perm_pvalue']:.4f}")

T1 = pd.DataFrame(rows)
T1[['label','domain','m','kappa','n_cross','copy_density','triple_fraction',
    'loc_x','perfect_x','perfect_over_loc','permuted_over_loc_mean',
    'permuted_over_loc_lo','permuted_over_loc_hi','structural_share',
    'structural_share_lo','structural_share_hi','perm_pvalue',
    'mindeg_over_loc','mindeg_capture','firstedge_x']].round(4)

In [ ]:
print('localization  : %.2fx - %.2fx  (mean %.2f), log-log slope in m = %.3f'
      % (T1.loc_x.min(), T1.loc_x.max(), T1.loc_x.mean(),
         np.polyfit(np.log(T1.m), np.log(T1.loc_x), 1)[0]))
print('predictor/loc : %.2fx - %.2fx  (mean %.2f)'
      % (T1.perfect_over_loc.min(), T1.perfect_over_loc.max(), T1.perfect_over_loc.mean()))
print('permuted/loc  : %.3f - %.3f' % (T1.permuted_over_loc_mean.min(), T1.permuted_over_loc_mean.max()))
print('struct share  : %.3f - %.3f  (mean %.3f)'
      % (T1.structural_share.min(), T1.structural_share.max(), T1.structural_share.mean()))
print('min-degree captures: %.3f of the achievable gain on average' % T1.mindeg_capture.mean())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].scatter(T1.m, T1.loc_x, c='steelblue'); ax[0].set_xscale('log')
ax[0].set_xlabel('m'); ax[0].set_ylabel('localization speedup'); ax[0].set_ylim(0, None)
ax[0].set_title('localization: a constant, not an exponent')
ax[1].scatter(T1.copy_density, T1.perfect_over_loc, c='crimson')
ax[1].set_xscale('log'); ax[1].set_xlabel('copy density  #cross / m')
ax[1].set_ylabel('predictor gain over localization')
ax[1].set_title('predictor value falls as copies saturate')
plt.tight_layout(); plt.show()

## 4. Validation — do the analytic probabilities match a real sampler?

Every number above is an exact sum over the sampling distribution rather than a simulation, which is
what makes them noise-free. A referee is entitled to ask whether the sum is the right one. Here we
run the actual weighted sampler — draw a base edge, draw a closing vertex, verify the cross triangle
— and compare the hit rate to the analytic value.

In [ ]:
val = []
for name in list(HG)[:4]:
    hg = HG[name]
    for tag, w in [('perfect', hg.w_perfect()), ('uniform', hg.w_uniform())]:
        an = hg.p_succ(w)
        mc = monte_carlo_check(hg, w, n_samples=40_000, seed=7)
        val.append(dict(label=name, predictor=tag, analytic=an, monte_carlo=mc,
                        rel_err=abs(mc - an) / an if an else np.nan))
        print(f'{name:24s} {tag:8s} analytic={an:.6f}  MC={mc:.6f}  rel.err={val[-1]["rel_err"]:.3f}')
V = pd.DataFrame(val)
V.round(5)

Relative errors of a few percent are the Monte-Carlo noise at 40k samples; systematic
disagreement would mean the canonical-path accounting is wrong. Raise `n_samples` if any row looks
off.

## 5. Table 2 — a learned predictor, strict leave-one-graph-out

This is now the crux. Notebook 2 found that on this primitive a shuffled weight distribution buys
nothing, so the entire benefit rests on identifying which edges bear cross-triangles. That is only
useful if a cheap model can identify them **on a graph it was not trained on**.

Three feature blocks, each standardized per graph:

* **A** — degrees and core numbers (the Array paper's block, transferred verbatim);
* **AB** — A plus cheap higher-order features: how many hyperedges contain each endpoint, how many
  contain the edge, $\sum_{h \ni u,v}(|h|-2)$, and the largest such hyperedge;
* **ABC** — AB plus the covered-neighbour count $|\bigcup_{h \ni u,v} h| - 2$.

Block C deserves a sentence in the paper: it is computable from hyperedge membership alone, without
any neighbourhood intersection, so it costs a preprocessing pass and no triangle work — but it is
per-pair state, so it is heavier than A or B in a true one-pass setting. Reporting all three keeps
that trade-off explicit.

In [ ]:
BLOCKS = ['A', 'AB', 'ABC']
LAMBDAS = [0.1, 1.0, 10.0]
names = list(HG)

pairs = []
for blk in BLOCKS:
    for lam in LAMBDAS:
        for tr, te in itertools.permutations(names, 2):
            mdl = ridge_fit(HG[tr].F[blk], HG[tr].y, lam=lam)
            w = ridge_predict_weights(mdl, HG[te].F[blk])
            hg = HG[te]
            p_loc = hg.p_succ(hg.w_uniform())
            s_learn = hg.p_succ(w) / p_loc
            s_perf = hg.p_succ(hg.w_perfect()) / p_loc
            gain = s_perf - 1.0
            pairs.append(dict(
                block=blk, lam=lam, train=tr, test=te,
                learned_over_loc=s_learn, perfect_over_loc=s_perf,
                capture=(s_learn - 1) / gain if abs(gain) > 1e-12 else np.nan,
                pred_corr=float(np.corrcoef(np.log1p(w), hg.y)[0, 1]),
            ))
P = pd.DataFrame(pairs)

def boot_ci(x, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]
    if not len(x):
        return (np.nan, np.nan)
    s = rng.choice(x, size=(B, len(x)), replace=True).mean(1)
    return float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5))

summ = []
for blk in BLOCKS:
    for lam in LAMBDAS:
        d = P[(P.block == blk) & (P.lam == lam)]
        lo, hi = boot_ci(d.capture)
        summ.append(dict(block=blk, lam=lam, n_pairs=len(d),
                         capture_mean=d.capture.mean(), capture_lo=lo, capture_hi=hi,
                         corr_mean=d['pred_corr'].mean(),
                         learned_over_loc_mean=d.learned_over_loc.mean()))
T2 = pd.DataFrame(summ)
T2.round(4)

In [ ]:
best = T2.sort_values('capture_mean').iloc[-1]
print(f"best block {best.block} (lambda={best.lam}): recovers "
      f"{best.capture_mean:.3f} of the achievable gain, 95% CI "
      f"[{best.capture_lo:.3f}, {best.capture_hi:.3f}] over {int(best.n_pairs)} ordered pairs; "
      f"mean prediction correlation {best.corr_mean:.3f}")
print('\nper-test-graph capture, best block:')
d = P[(P.block == best.block) & (P.lam == best.lam)]
print(d.groupby('test')[['capture','pred_corr','learned_over_loc']].mean().round(3).to_string())

## 6. Table 3 — robustness of our own choices

Two checks a referee will want. First, that the notebook-2 numbers were not an artefact of capping
the number of simplices: rerun one large dataset at several caps and show the reported ratios are
stable. Second, that the triple subsampling does not move the ratios: rerun one dataset at two
different `max_triples` and seeds.

In [ ]:
BIG = 'congress-bills' if 'congress-bills' in HYPER else list(HYPER)[-1]
rows = []
for cap in [3_000, 10_000, 30_000, len(HYPER[BIG])]:
    hg = CrossHG(HYPER[BIG][:cap], max_size=25, max_triples=MAX_TRIPLES, seed=0, verbose=False)
    r = evaluate(hg, n_perm=15, seed=0)
    r['cap'] = cap
    rows.append(r)
    print(f'cap={cap:>7d}  m={r["m"]:>7d}  loc={r["loc_x"]:.2f}x  '
          f'pred={r["perfect_over_loc"]:.3f}x  share={r["structural_share"]:.3f}')
T3 = pd.DataFrame(rows)[['cap','m','n_cross','copy_density','triple_fraction',
                         'loc_x','perfect_over_loc','permuted_over_loc_mean','structural_share']]

rows = []
for mt, sd in [(1_000_000, 1), (2_000_000, 2), (4_000_000, 3)]:
    hg = CrossHG(HYPER[BIG], max_size=25, max_triples=mt, seed=sd, verbose=False)
    r = evaluate(hg, n_perm=10, seed=0)
    r['max_triples'], r['seed'] = mt, sd
    rows.append(r)
T3b = pd.DataFrame(rows)[['max_triples','seed','triple_fraction','loc_x',
                          'perfect_over_loc','structural_share']]
print()
print(T3.round(4).to_string(index=False))
print()
print(T3b.round(4).to_string(index=False))

## 7. The synthetic control: the machinery *can* produce an exponent

The contrast that makes the real-data flatness meaningful. On a separable family — disjoint large
hyperedges as copy-free cliques, plus shallow cross gadgets — the same localization quantity should
grow polynomially in $m$.

In [ ]:
def cliques_plus_cross(n_cliques, clique_size, n_gadgets, seed=0):
    rng = np.random.default_rng(seed)
    H, nxt, cliques = [], 0, []
    for _ in range(n_cliques):
        c = list(range(nxt, nxt + clique_size)); nxt += clique_size
        H.append(tuple(c)); cliques.append(c)
    for _ in range(n_gadgets):
        c = cliques[rng.integers(len(cliques))]
        u, v = rng.choice(c, size=2, replace=False)
        w = nxt; nxt += 1
        H.append((int(u), w)); H.append((int(v), w))
    return H

rows = []
for cs in [20, 30, 40, 60, 80]:
    hg = CrossHG(cliques_plus_cross(6, cs, 6 * cs, seed=1), max_size=200, verbose=False)
    r = evaluate(hg, n_perm=15, seed=0); r['label'] = f'cliques(r={cs})'
    rows.append(r)
S = pd.DataFrame(rows)
b_loc = np.polyfit(np.log(S.m), np.log(S.loc_x), 1)[0]
print(f'separable family: localization grows as m^{b_loc:.3f} '
      f'(prediction-free exponent rho(K3)-1 = 0.5)')
print(f'real hypergraphs: localization grows as m^'
      f'{np.polyfit(np.log(T1.m), np.log(T1.loc_x), 1)[0]:.3f}')

plt.figure(figsize=(5.6, 3.8))
plt.loglog(S.m, S.loc_x, 'o-', label=f'separable synthetic (m^{b_loc:.2f})')
plt.scatter(T1.m, T1.loc_x, c='crimson', marker='x', s=60, label='real hypergraphs (flat)')
plt.xlabel('m'); plt.ylabel('localization speedup'); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()
S[['label','m','n_cross','loc_x','perfect_over_loc','permuted_over_loc_mean',
   'structural_share']].round(3)

## 8. Save everything

In [ ]:
T1.to_csv('table1_main.csv', index=False)
P.to_csv('table2_pairs_raw.csv', index=False)
T2.to_csv('table2_learned.csv', index=False)
T3.to_csv('table3_cap_sensitivity.csv', index=False)
T3b.to_csv('table3b_triple_sensitivity.csv', index=False)
S.to_csv('table4_synthetic.csv', index=False)
V.to_csv('validation_montecarlo.csv', index=False)
print('saved 7 csv files')
print()
print('=== headline numbers ===')
print(f'datasets: {len(T1)} across {T1.domain.nunique()} domains, m from {T1.m.min()} to {T1.m.max()}')
print(f'localization      {T1.loc_x.min():.2f}x - {T1.loc_x.max():.2f}x, slope in m '
      f'{np.polyfit(np.log(T1.m), np.log(T1.loc_x), 1)[0]:+.3f}')
print(f'perfect predictor {T1.perfect_over_loc.min():.2f}x - {T1.perfect_over_loc.max():.2f}x on top')
print(f'permutation null  {T1.permuted_over_loc_mean.min():.3f} - '
      f'{T1.permuted_over_loc_mean.max():.3f}  (max p-value {T1.perm_pvalue.max():.4f})')
print(f'structural share  {T1.structural_share.min():.3f} - {T1.structural_share.max():.3f} '
      f'(Array paper, ordinary triangles: 0.093)')
print(f'learned predictor recovers {best.capture_mean:.3f} '
      f'[{best.capture_lo:.3f}, {best.capture_hi:.3f}] with block {best.block}')
print(f'min-degree heuristic recovers {T1.mindeg_capture.mean():.3f}')

## 9. Where each table goes in the paper

* **Table 1** → §5.1, the headline. The two facts: localization is a constant across five domains and
  two orders of magnitude in $m$, and the permutation null sits at or below 1.
* **Figure (right panel of §3)** → §5.2, the proposed second diagnostic: predictor value falls as
  copy density rises. State it as an observed regularity over nine graphs, not a law.
* **Table 2** → §5.3, and it decides the paper's practical advice. If `capture_mean` is high, the
  message is "for higher-order triadic closure, unlike ordinary triangles, train a predictor — and
  here are the features". If it is low, the message is "the signal is real but not cheaply learnable",
  which is a weaker but still honest and publishable finding.
* **Table 3** → appendix, robustness.
* **Table 4 + figure** → §5.4, the synthetic contrast that shows the flatness is a property of the
  data and not of the estimator.
* **Validation table** → appendix, to justify reporting exact probabilities rather than simulations.

Two things still to write by hand, not computable here: the delta paragraph against the localization
paper under review at DPD, and the related-work positioning against Tonic and the simplicial-closure
literature (Benson et al., PNAS 2018), which is where the cross-hyperedge triangle comes from as a
data-mining object.